# TCC - Análise Preditiva de Risco Acadêmico Escolar
## CRISP-DM: 5. Evaluation


## O que esta fase avalia

A avaliação verifica se a modelagem produziu sinais úteis para a escola:

- se a previsão de nota erra pouco;
- se o alerta de risco encontra alunos que realmente ficaram abaixo de 6.0;
- se o candidato escolhido supera baselines simples;
- se há disciplinas, séries ou faixas de nota com erro mais alto;
- se os resultados são coerentes o suficiente para seguir para uso institucional.

Nesta fase, a avaliação deve ser lida como verificação do desempenho no conjunto de teste final, depois que a escolha dos candidatos já foi feita na validação.


## Carregar bibliotecas e localizar o projeto

In [ ]:
from pathlib import Path
import re
import sys

import pandas as pd
from IPython.display import display

In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "school_predictor").exists() and (candidate / "artifacts").exists():
            return candidate.resolve()
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## Importar caminhos e execução opcional

A avaliação normalmente usa artefatos já gerados pelas execuções anteriores da pipeline. Se eles não existirem, é possível rodar a pipeline localmente ativando uma flag.



In [ ]:
from school_predictor.pipeline.config import PipelinePaths
from school_predictor.pipeline.orchestration import resolve_mode_settings, run_real_pipeline

## Escolher modo avaliado

Os dois modos podem ser avaliados:

- `previsao_nota`: foco principal na nota prevista.
- `alerta_risco`: foco principal no alerta pedagógico.

Ambos geram métricas de regressão e classificação, pois a pipeline calcula os dois sinais de forma conjunta.

O que muda entre os modos é principalmente o `min_history`, o volume da base avaliada e os artefatos salvos. A lógica de avaliação continua analisando nota prevista e risco previsto no teste final.



In [ ]:
MODE = "previsao_nota"  # alternativas: "previsao_nota" ou "alerta_risco"
MODE, MIN_HISTORY = resolve_mode_settings(MODE, min_history=None)
paths = PipelinePaths.from_project_root(PROJECT_ROOT, min_history=MIN_HISTORY, mode=MODE)

paths.output_dir

### O que este modo muda na avaliação

A célula acima define qual conjunto de artefatos será lido. Isso importa porque `previsao_nota` e `alerta_risco` podem ter recortes de base e candidatos vencedores diferentes, mesmo compartilhando a mesma lógica geral de avaliação.



## Opcional: gerar artefatos se ainda não existirem

Por padrão, este notebook não treina nem salva nada. Se os arquivos de avaliação não existirem localmente, altere `RUN_PIPELINE_IF_MISSING = True`.


In [ ]:
RUN_PIPELINE_IF_MISSING = False

required_artifacts = [
    paths.predictions_csv,
    paths.metrics_txt,
    paths.error_by_subject_csv,
    paths.error_by_series_csv,
    paths.error_by_band_csv,
]

missing = [path for path in required_artifacts if not path.exists()]
if missing and RUN_PIPELINE_IF_MISSING:
    run_real_pipeline(project_root=PROJECT_ROOT, mode=MODE, min_history=MIN_HISTORY)
elif missing:
    print("Artefatos ausentes. Rode a pipeline pela CLI ou ative RUN_PIPELINE_IF_MISSING localmente.")
    for path in missing:
        print(path)

## Carregar artefatos de avaliação

A tabela de predições contém o teste final. As tabelas de erro agrupam os resultados do mesmo conjunto de teste por disciplina, série e faixa de nota.

Em outras palavras: a validação serviu para escolher os candidatos; o que este notebook inspeciona agora é o desempenho final no ano de teste.



### Exemplo fixo dos artefatos usados na avaliação

A avaliação desta fase consulta principalmente estes artefatos técnicos já gerados para o modo escolhido:

<div style="font-size: 1em; overflow-x: auto;">

| modo | arquivo | papel_na_avaliacao |
| --- | --- | --- |
| previsao_nota | student_prediction_predictions.csv | comparar nota real, nota prevista, baseline de referência e risco previsto no teste |
| previsao_nota | student_prediction_report.txt | mostrar ranking de candidatos e métricas finais |
| previsao_nota | error_analysis_by_subject.csv | localizar disciplinas com maior erro |
| previsao_nota | error_analysis_by_series.csv | comparar desempenho por série |
| previsao_nota | error_analysis_by_score_band.csv | comparar erro por faixa de nota |
| alerta_risco | student_prediction_predictions.csv | avaliar classificação de risco no conjunto de teste |

</div>


In [ ]:
predictions = pd.read_csv(paths.predictions_csv, encoding="utf-8", low_memory=False)
error_by_subject = pd.read_csv(paths.error_by_subject_csv, encoding="utf-8", low_memory=False)
error_by_series = pd.read_csv(paths.error_by_series_csv, encoding="utf-8", low_memory=False)
error_by_band = pd.read_csv(paths.error_by_band_csv, encoding="utf-8", low_memory=False)
report_text = paths.metrics_txt.read_text(encoding="utf-8")

pd.DataFrame([{
    "modo": MODE,
    "predicoes": len(predictions),
    "disciplinas_avaliadas": len(error_by_subject),
    "series_avaliadas": len(error_by_series),
    "faixas_avaliadas": len(error_by_band),
}])

### O que realmente entrou na avaliação

Depois desta etapa, o notebook deixa de olhar para a base de treino e passa a olhar para os artefatos do teste final. A tabela de predições representa o que o modelo fez no ano mais recente; as tabelas agrupadas mostram onde ele foi melhor ou pior.

Essa diferença é crucial: aqui já não estamos escolhendo candidato. Estamos auditando o comportamento do vencedor.



## Interpretação das métricas

<div style="font-size: 1em; overflow-x: auto;">

| Métrica | Onde aparece | Como interpretar |
|---|---|---|
| MAE | Regressão | erro médio absoluto da nota; quanto menor, melhor |
| RMSE | Regressão | penaliza erros grandes; quanto menor, melhor |
| Acerto <= 0.5 | Regressão | percentual de previsões com erro de até meio ponto |
| Precision | Classificação | dos alertas emitidos, quantos eram risco real |
| Recall | Classificação | dos riscos reais, quantos foram encontrados |
| F1 | Classificação | equilíbrio entre precision e recall |

</div>

No contexto pedagógico, recall baixo pode ser perigoso porque indica que alunos em risco deixaram de ser alertados. Precision muito baixo também é ruim porque gera muitos alertas falsos e pode cansar a equipe.

Por isso, este projeto não usa apenas acurácia geral. Em bases escolares, um modelo pode acertar muitos casos triviais e ainda assim falhar justamente nos alunos que mais importam.



## Extrair resumo técnico do relatório

O relatório técnico salvo pela pipeline registra quem venceu na validação e quais métricas esse vencedor obteve no teste final.


### Exemplo fixo do resumo técnico

Abaixo está um resumo mais legível das métricas finais e da referência usada para comparação:

<div style="font-size: 1em; overflow-x: auto;">

| modo | tarefa | metrica_principal | modelo_final | valor_modelo | baseline_referencia | valor_baseline |
| --- | --- | --- | --- | --- | --- | --- |
| previsao_nota | regressão | MAE | RandomForest | 0.7659 | ultima_nota | 0.9138 |
| previsao_nota | classificação | F1 | Baseline media_duas_ultimas | 0.7350 | ultima_nota | 0.7252 |
| alerta_risco | regressão | MAE | HistGradientBoosting | 0.8140 | ultima_nota | 0.9405 |
| alerta_risco | classificação | F1 | HistGradientBoosting | 0.7436 | ultima_nota | 0.7243 |

</div>


In [ ]:
def parse_metric_line(pattern: str, text: str) -> float | None:
    match = re.search(pattern, text)
    return float(match.group(1)) if match else None


selected_candidates = pd.DataFrame([
    {"tipo": "regressao", "candidato": re.search(r"candidato selecionado para regressao: ([^\n]+)", report_text).group(1)},
    {"tipo": "classificacao", "candidato": re.search(r"candidato selecionado para classificacao: ([^\n]+)", report_text).group(1)},
])

metric_summary = pd.DataFrame([
    {"grupo": "modelo_regressao", "metrica": "mae", "valor": parse_metric_line(r"modelo mae: ([0-9.]+)", report_text)},
    {"grupo": "modelo_regressao", "metrica": "rmse", "valor": parse_metric_line(r"modelo rmse: ([0-9.]+)", report_text)},
    {"grupo": "modelo_regressao", "metrica": "acerto_ate_0_5", "valor": parse_metric_line(r"modelo acerto <= 0.5: ([0-9.]+)", report_text)},
    {"grupo": "baseline_regressao", "metrica": "mae", "valor": parse_metric_line(r"baseline mae: ([0-9.]+)", report_text)},
    {"grupo": "baseline_regressao", "metrica": "rmse", "valor": parse_metric_line(r"baseline rmse: ([0-9.]+)", report_text)},
    {"grupo": "baseline_regressao", "metrica": "acerto_ate_0_5", "valor": parse_metric_line(r"baseline acerto <= 0.5: ([0-9.]+)", report_text)},
    {"grupo": "modelo_classificacao", "metrica": "precision", "valor": parse_metric_line(r"modelo precision: ([0-9.]+)", report_text)},
    {"grupo": "modelo_classificacao", "metrica": "recall", "valor": parse_metric_line(r"modelo recall: ([0-9.]+)", report_text)},
    {"grupo": "modelo_classificacao", "metrica": "f1", "valor": parse_metric_line(r"modelo f1: ([0-9.]+)", report_text)},
    {"grupo": "baseline_classificacao", "metrica": "precision", "valor": parse_metric_line(r"baseline precision: ([0-9.]+)", report_text)},
    {"grupo": "baseline_classificacao", "metrica": "recall", "valor": parse_metric_line(r"baseline recall: ([0-9.]+)", report_text)},
    {"grupo": "baseline_classificacao", "metrica": "f1", "valor": parse_metric_line(r"baseline f1: ([0-9.]+)", report_text)},
])

display(selected_candidates)
metric_summary

### Como ler o resumo técnico extraido do relatório

A saída acima concentra o que a pipeline registrou como desempenho final no teste. Ela junta, em formato compacto, o candidato escolhido e as métricas do modelo e do baseline de referência final. Esse baseline de comparação no teste é `ultima_nota`, mesmo quando outro baseline tenha vencido a validação.

Na prática, esta é a visão que responde duas perguntas de defesa:
- quem venceu na validação?
- no teste final, esse vencedor realmente entregou ganho útil?



## Avaliação direta na tabela de predições

Este bloco recalcula métricas simples a partir do arquivo de predições para conferir os resultados do relatório.

A ideia aqui não é substituir o relatório técnico, mas mostrar que os valores podem ser reproduzidos diretamente do conjunto de teste salvo em CSV.



In [ ]:
tp = ((predictions["target_risco_proxima"] == 1) & (predictions["predicted_risk_flag"] == 1)).sum()
fp = ((predictions["target_risco_proxima"] == 0) & (predictions["predicted_risk_flag"] == 1)).sum()
fn = ((predictions["target_risco_proxima"] == 1) & (predictions["predicted_risk_flag"] == 0)).sum()

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

direct_metrics = pd.DataFrame([{
    "mae_recalculado": predictions["absolute_error"].mean(),
    "rmse_recalculado": (predictions["signed_error"].pow(2).mean()) ** 0.5,
    "acerto_ate_0_5_recalculado": (predictions["absolute_error"] <= 0.5).mean(),
    "precision_risco_recalculada": precision,
    "recall_risco_recalculado": recall,
    "f1_risco_recalculado": f1,
    "acuracia_risco_recalculada": predictions["risk_hit"].mean(),
    "taxa_risco_real": predictions["target_risco_proxima"].mean(),
    "taxa_risco_previsto": predictions["predicted_risk_flag"].mean(),
}])

direct_metrics

### O que esta conferência prova

A tabela recalculada mostra que as métricas podem ser reproduzidas diretamente do CSV de predições. Isso é importante porque demonstra rastreabilidade: o relatório técnico não é uma caixa-preta separada dos dados.

Quando os números batem, a avaliação ganha credibilidade metodológica.





## Amostra segura de predições

A amostra abaixo usa os nomes já pseudonimizados da base pública e mostra apenas os campos pedagógicos essenciais. Ainda assim, evite salvar outputs executados no GitHub quando não forem necessários.


### Exemplo fixo da tabela de predições

Esta amostra mostra como uma linha do teste é avaliada na prática, usando os nomes já pseudonimizados da base pública:

<div style="font-size: 0.9em; overflow-x: auto;">

| NomeAluno | NomePeriodo | NomeSerie | NomeDisciplina | NomeEtapa | ValorMedia | target_nota_proxima | predicted_next_grade | baseline_next_grade | absolute_error | signed_error | target_risco_proxima | predicted_risk_flag | risk_hit |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Aluno 01790 | 2025 | 1ª Série | Arte | 2º BIMESTRE | 8.8 | 8.1 | 8.9894 | 8.8 | 0.8894 | 0.8894 | 0 | 0 | 1 |
| Aluno 01790 | 2025 | 1ª Série | Arte | 3º BIMESTRE | 8.1 | 10.0 | 9.0035 | 8.1 | 0.9965 | -0.9965 | 0 | 0 | 1 |
| Aluno 01790 | 2025 | 1ª Série | Biologia | 2º BIMESTRE | 7.0 | 7.4 | 7.7188 | 7.0 | 0.3188 | 0.3188 | 0 | 0 | 1 |
| Aluno 01790 | 2025 | 1ª Série | Biologia | 3º BIMESTRE | 7.4 | 7.0 | 8.0697 | 7.4 | 1.0697 | 1.0697 | 0 | 0 | 1 |
| Aluno 01790 | 2025 | 1ª Série | Biologia 1 | 2º BIMESTRE | 7.0 | 5.6 | 7.4285 | 7.0 | 1.8285 | 1.8285 | 1 | 0 | 0 |

</div>


In [ ]:
public_prediction_columns = [
    "NomePeriodo", "NomeSerie", "NomeDisciplina", "NomeEtapa",
    "ValorMedia", "target_nota_proxima", "predicted_next_grade",
    "absolute_error", "target_risco_proxima", "predicted_risk_flag", "risk_hit",
]

predictions[[column for column in public_prediction_columns if column in predictions.columns]].head(10)

### O que esta amostra de predições revela

Agora você enxerga uma linha real de avaliação: nota atual, alvo da próxima nota, nota prevista, baseline de referência, erro absoluto, erro assinado, risco real e risco previsto. Isso ajuda a banca a sair do nível abstrato das métricas e ver como o sistema se comporta caso a caso.


## Análise de erro por disciplina

Esta tabela mostra onde o erro médio de nota foi maior. Ela ajuda a identificar disciplinas que podem precisar de leitura mais cuidadosa, mais contexto ou ajustes futuros. Além do `mae`, a saída completa também traz `rmse`, `mean_signed_error`, `risk_accuracy` e `actual_risk_rate`.



### Exemplo fixo do erro por disciplina

As disciplinas abaixo são as que concentraram maior erro médio absoluto no modo `previsao_nota`:

<div style="font-size: 1em; overflow-x: auto;">

| NomeDisciplina | samples | mae | risk_accuracy |
| --- | --- | --- | --- |
| Matemática 3 | 163 | 1.5887 | 0.6871 |
| Matemática 4 e 5 | 163 | 1.4575 | 0.8589 |
| Biologia 2 | 521 | 1.3515 | 0.7006 |
| Biologia 1 | 521 | 1.1428 | 0.7831 |
| Matemática 1 e 2 | 163 | 1.1357 | 0.8957 |

</div>


In [ ]:
error_by_subject.head(15)

### O que o erro por disciplina mostra

Quando esta tabela aparece, a avaliação deixa de ser global. Ela passa a apontar onde o modelo sofre mais. Isso é essencial em contexto escolar, porque um MAE médio bom pode esconder disciplinas específicas com comportamento bem pior.




## Análise de erro por série

Avaliação por série ajuda a saber se o modelo funciona melhor em certos anos escolares e pior em outros. As métricas agrupadas também permitem ver se há tendência de superestimar ou subestimar notas em séries específicas.



### Exemplo fixo do erro por série

Aqui a avaliação mostra quais séries ficaram mais difíceis de prever:

<div style="font-size: 1em; overflow-x: auto;">

| NomeSerie | samples | mae | risk_accuracy |
| --- | --- | --- | --- |
| 2ª Série | 4756 | 0.8739 | 0.8522 |
| 9º Ano | 3698 | 0.8701 | 0.8437 |
| 3ª Série | 5053 | 0.8398 | 0.8698 |
| 1ª Série | 5432 | 0.8294 | 0.8575 |
| 6º Ano | 2610 | 0.5864 | 0.9479 |

</div>


In [ ]:
error_by_series.head(15)

### O que o erro por série mostra

Aqui a pergunta muda de disciplina para etapa escolar. A saída ajuda a enxergar se o sistema funciona de modo desigual entre anos escolares, o que pode indicar necessidade de novos atributos ou leitura pedagógica mais cuidadosa em certas séries.


## Análise de erro por faixa de nota

A avaliação por faixa mostra se o modelo erra mais em alunos de nota baixa, média ou alta. Isso é importante porque erros em notas próximas de 6.0 podem mudar a decisão de alerta. Nesta análise, a faixa é construída a partir da nota real futura (`target_nota_proxima`), não da nota prevista.



### Exemplo fixo do erro por faixa de nota

Esta tabela ajuda a ver se o erro muda quando a nota real está em faixas diferentes:

<div style="font-size: 1em; overflow-x: auto;">

| score_band | samples | mae | risk_accuracy |
| --- | --- | --- | --- |
| abaixo_6 | 5756 | 1.031 | 0.751 |
| 6_a_6_99 | 3063 | 0.877 | 0.6941 |
| 7_a_7_99 | 3476 | 0.766 | 0.8708 |
| 8_a_8_99 | 4220 | 0.6858 | 0.9491 |
| 9_ou_mais | 10012 | 0.6134 | 0.9916 |

</div>


In [ ]:
error_by_band

### O que a faixa de nota revela

Esta análise mostra se o modelo erra mais justamente nos alunos mais sensíveis. Em projetos educacionais, isso importa muito, porque pequenas diferenças perto da linha de 6.0 podem mudar completamente a interpretação do risco.



## Matriz simples de risco

A matriz abaixo compara risco real e risco previsto. Ela ajuda a visualizar falsos positivos e falsos negativos. No notebook, essa matriz é reconstruída diretamente a partir de `target_risco_proxima` e `predicted_risk_flag` salvos no CSV de predições.

Leitura rápida:
- falso negativo é aluno em risco real que o sistema não alertou;
- falso positivo é aluno alertado que não entrou em risco na nota seguinte.

No contexto escolar deste projeto, falsos negativos costumam ser mais preocupantes do que falsos positivos moderados.




### Exemplo fixo da matriz de risco

A matriz abaixo resume acertos e erros da classificação de risco no teste:

<div style="font-size: 1em; overflow-x: auto;">

| risco_real | previsto_sem_risco | previsto_com_risco |
| --- | --- | --- |
| real_sem_risco | 19086 | 1685 |
| real_com_risco | 1433 | 4323 |

</div>


In [ ]:
risk_matrix = pd.crosstab(
    predictions["target_risco_proxima"],
    predictions["predicted_risk_flag"],
    rownames=["risco_real"],
    colnames=["risco_previsto"],
    margins=True,
)

risk_matrix

### O que a matriz de risco deixa visível

A matriz transforma a classificação em contagens concretas. Depois dela, fica fácil explicar quantos casos foram:
- acertos de risco;
- acertos de não risco;
- falsos positivos;
- falsos negativos.

Essa leitura ajuda a traduzir precision, recall e F1 em impacto operacional para a escola.




## Critérios para dizer se está bom

Não existe um único número universal. Para este projeto, a avaliação deve considerar:

- MAE menor que o baseline indica ganho real sobre regra simples.
- RMSE muito maior que MAE indica alguns erros grandes que precisam ser investigados.
- Acerto até 0.5 ponto ajuda a saber se a previsão é pedagogicamente próxima.
- F1 maior que o baseline indica que a classificação aprendeu algo além de regras simples.
- Recall baixo exige cautela, pois significa deixar alunos em risco sem alerta.
- Precision baixo exige cautela, pois gera muitos alertas falsos para professores e coordenação.

Também é importante separar duas ideias: o baseline vencedor na validação pode ser uma regra simples como `media_duas_ultimas`, mas a referência fixa de comparação no teste final continua sendo `ultima_nota`.






## Resultado da fase

Ao final da avaliação, o projeto deve conseguir explicar:

- qual candidato foi escolhido na validação;
- se ele superou os baselines no teste final;
- quais foram os erros médios de nota;
- como o alerta de risco se comportou;
- onde o modelo erra mais;
- quais cuidados de interpretação devem acompanhar a passagem para uso institucional.

Essas conclusões seguem para a fase de Deployment, onde os resultados passam a ser transformados em relatórios e dashboard para os perfis da escola.

A avaliação, portanto, não responde apenas se o modelo acertou ou errou. Ela responde se o sinal produzido é suficientemente confiável para seguir para uso institucional sem criar falsa segurança nem ruído excessivo.





## Leitura didática da avaliação

O resultado final desta fase não é apenas dizer que o modelo ficou "bom" ou "ruim". O que sai daqui é uma interpretação defensável sobre três dimensões ao mesmo tempo:
- ganho sobre baseline;
- comportamento por grupos críticos;
- utilidade institucional do sinal gerado.


